# Cost Architecture — Hands-On

**LLM Engineering · Domain 8 · Roadmap Week 22**

Offline notebook: token/cost ledger, exact+semantic cache, confidence-gated model cascade, routing budgets.

## 0. Setup

In [ ]:
%pip install -q numpy
import re, numpy as np
rng = np.random.RandomState(47)
print("ok")

## 1. Token and cost ledger

In [ ]:
PRICES = {"cheap": (0.10, 0.30), "strong": (2.00, 8.00)}
def tokens(text): return max(1, len(text.split()) * 4 // 3)
def price(model, prompt, output):
    it, ot = tokens(prompt), tokens(output)
    pin, pout = PRICES[model]
    return {"model": model, "input_tokens": it, "output_tokens": ot, "cost": (it*pin + ot*pout)/1_000_000}
print(price("cheap", "classify this ticket", "billing"))

## 2. Exact plus semantic cache

In [ ]:
def norm(s): return re.sub(r"\s+", " ", s.lower()).strip()
def embed(s):
    v = np.zeros(12)
    for ch in norm(s): v[ord(ch)%12] += 1
    return v/(np.linalg.norm(v)+1e-9)
def cos(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))
class Cache:
    def __init__(self): self.exact = {}; self.semantic = []
    def put(self, q, a, version="v1"):
        key = (version, norm(q)); self.exact[key] = a; self.semantic.append((version, embed(q), a))
    def get(self, q, version="v1", threshold=.92):
        key = (version, norm(q))
        if key in self.exact: return "exact", self.exact[key]
        scored = [(cos(embed(q), v), a) for ver, v, a in self.semantic if ver == version]
        return ("semantic", max(scored)[1]) if scored and max(scored)[0] >= threshold else ("miss", None)
cache = Cache(); cache.put("How reset password?", "Use reset link")
print(cache.get("how reset password")); print(cache.get("refund policy"))

## 3. Router and confidence cascade

In [ ]:
def cheap_model(task):
    easy = any(w in task for w in ["classify", "extract", "summarize"])
    return {"answer": "cheap answer", "confidence": .86 if easy else .45, **price("cheap", task, "cheap answer")}
def strong_model(task): return {"answer": "strong answer", "confidence": .94, **price("strong", task, "strong answer")}
def cascade(task, gate=.75):
    first = cheap_model(task)
    if first["confidence"] >= gate: return first | {"route": "cheap"}
    second = strong_model(task); second["cost"] += first["cost"]
    return second | {"route": "escalated"}
for t in ["classify a refund ticket", "solve hard multi step reasoning"]: print(cascade(t))

## 4. Budget enforcement

In [ ]:
ledger = [{"tenant":"a", "cost": .03}, {"tenant":"a", "cost": .04}, {"tenant":"b", "cost": .20}]
budgets = {"a": .10, "b": .15}
spent = {t: sum(x["cost"] for x in ledger if x["tenant"] == t) for t in budgets}
print(spent)
print({t: spent[t] <= budgets[t] for t in budgets})

## 5. Token reduction effect

In [ ]:
prompt = "policy chunk " * 800
compressed = "policy summary " * 120
out = "answer " * 50
before = price("strong", prompt, out)["cost"]
after = price("strong", compressed, out)["cost"]
print("before", round(before, 5), "after", round(after, 5), "saving", round((before-after)/before, 2))

## 6. Batch API discount simulation

In [ ]:
requests = [price("cheap", "summarize ticket "*rng.randint(5,20), "summary") for _ in range(20)]
regular = sum(r["cost"] for r in requests)
batch = regular * 0.5
print("regular", round(regular, 6), "batch", round(batch, 6))

## 7. Exercises
1. Add cache TTL and invalidate old prompt versions.
2. Add a safety-risk route that always uses the strong model.
3. Compute cache hit rate and savings over 100 simulated requests.
4. Add per-feature budgets to the gateway ledger.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Cost Architecture`
- Builds on: `02 Literature Notes/LLM Engineering/Reasoning Models`